[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/01_text_baseline/01_text_baseline_solutions.ipynb)

# 01. 텍스트 분류 기준선 — 연습 문제 해설

[01_text_baseline.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/01_text_baseline/01_text_baseline.ipynb) 끝의
연습 문제 6개에 대한 정답 코드와 해설입니다. **먼저 직접 시도해본 뒤** 참고하세요.

> 아래 숫자는 `random_state=42` 기준입니다. 여러분의 실행 결과와 소수점 이하가 다를 수 있습니다.
>
> **문제 4와 5가 가장 오래 걸립니다**(각각 2~4분). 나머지는 문제당 1분 안쪽입니다.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q pandas scikit-learn matplotlib

YNAT = "https://raw.githubusercontent.com/KLUE-benchmark/KLUE/main/klue_benchmark/ynat-v1.1"

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline

RANDOM_STATE = 42

전체 = pd.read_json(f"{YNAT}/ynat-v1.1_train.json")[["title", "label"]]
data = 전체.sample(20_000, random_state=RANDOM_STATE).reset_index(drop=True)

X, y = data["title"], data["label"]
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)


def vectorizer(**kwargs):
    """본문 8절에서 채택한 벡터화 설정."""
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3), **kwargs)


def 학습(Xa, ya, Xb, yb, **kwargs):
    """모델을 학습하고 (모델, 정확도)를 돌려준다."""
    m = make_pipeline(
        vectorizer(), LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, **kwargs)
    ).fit(Xa, ya)
    return m, accuracy_score(yb, m.predict(Xb))


기준모델, 기준정확도 = 학습(X_train, y_train, X_valid, y_valid)
print("학습", len(X_train), "· 검증", len(X_valid))
print("본문 8절의 기준 정확도: %.4f" % 기준정확도)

---

## 문제 1. `사회`를 빼면 정확도가 얼마나 오르나

본문 10절에서 `사회`의 재현율이 0.55로 유독 낮았습니다. 이 주제를 빼고 6개만 분류하면 어떻게 될까요.

In [ ]:
사회아님_train = y_train != "사회"
사회아님_valid = y_valid != "사회"

모델6, 정확도6 = 학습(
    X_train[사회아님_train], y_train[사회아님_train],
    X_valid[사회아님_valid], y_valid[사회아님_valid],
)

print("7개 주제 전부 : %.4f  (검증 %d건)" % (기준정확도, len(X_valid)))
print("6개 주제만    : %.4f  (검증 %d건)" % (정확도6, 사회아님_valid.sum()))

# 중요: 기존 7개 모델을 '사회가 아닌 행'에서만 채점하면?
print("\n7개 모델을 같은 %d건에서만 채점: %.4f"
      % (사회아님_valid.sum(),
         accuracy_score(y_valid[사회아님_valid], 기준모델.predict(X_valid[사회아님_valid]))))

**0.8455 → 0.9017. 5.6%p나 올랐습니다. 그런데 이것은 개선이 아닙니다.**

세 번째 줄이 그 이유를 보여줍니다. **기존 모델을 그대로 두고 채점 대상에서 `사회`만 빼도 0.8831**입니다.
모델은 손도 안 댔는데 3.8%p가 올랐습니다.

| | 정확도 | 무엇이 달라졌나 |
|---|---|---|
| 7개 주제, 전체 채점 | 0.8455 | — |
| 7개 주제, **`사회`만 빼고 채점** | 0.8831 | **모델은 그대로.** 어려운 문제만 뺐다 |
| 6개 주제로 다시 학습 | 0.9017 | 위에 더해, `사회`로 새어나가던 오답도 사라졌다 |

**올라간 5.6%p 중 3.8%p는 순전히 "어려운 문제를 시험에서 뺀 것"입니다.**
나머지 1.9%p만 실제로 문제가 쉬워진 몫입니다(`경제` 기사를 `사회`로 잘못 보낼 일이 없어졌으니까요).

**이 실험이 알려주는 것.** 정확도는 **어떤 데이터에서 쟀는지**에 따라 얼마든지 달라집니다.
서로 다른 두 숫자를 비교할 때는 **같은 대상을 채점했는지** 먼저 확인해야 합니다.
논문이나 블로그에서 "우리 모델이 0.90"이라는 숫자를 볼 때도 마찬가지입니다 —
**어떤 클래스를, 어떤 데이터에서** 쟀는지가 모델 구조보다 점수를 크게 움직입니다.

> **그럼 `사회`를 빼는 게 좋은 선택일까요?** 그건 **문제 정의의 문제**이지 성능의 문제가 아닙니다.
> 실제로 `사회` 기사를 분류할 필요가 없다면 빼는 것이 맞고, 필요하다면 점수가 낮아도 남겨야 합니다.
> **점수를 올리려고 클래스를 빼는 것**은 자기 자신을 속이는 일입니다.

---

## 문제 2. `min_df` / `max_df`로 피처 줄이기

In [ ]:
for min_df in [1, 2, 5]:
    for max_df in [1.0, 0.5]:
        pipe = make_pipeline(
            vectorizer(min_df=min_df, max_df=max_df),
            LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        ).fit(X_train, y_train)
        acc = accuracy_score(y_valid, pipe.predict(X_valid))
        n_features = pipe[:-1].transform(X_train[:1]).shape[1]
        print(f"min_df={min_df}, max_df={max_df}: 정확도 {acc:.4f}  피처 {n_features:,}개")

**피처를 1/5로 줄였는데 정확도는 오히려 조금 올랐습니다.**

| `min_df` | 정확도 | 피처 수 |
|---|---|---|
| 1 | 0.8455 | 132,843 |
| 2 | 0.8460 | 60,181 |
| 5 | **0.8468** | **24,350** |

- `min_df=5`는 "다섯 문서 미만에 나온 조각은 버린다"는 뜻입니다. 그렇게 드문 글자 조각은
  대개 외래어 표기나 특이한 고유명사라, 학습 데이터에서만 통하고 새 데이터에는 안 나옵니다
- `max_df=0.5`는 이 데이터에서 **아무것도 걸러내지 않았습니다**(피처 수가 그대로).
  전체 문서의 절반 이상에 나오는 글자 조각이 없기 때문입니다.
  문서가 긴 데이터(뉴스 본문·리뷰)에서는 효과가 있습니다

**정확도 차이는 0.0013으로, 본문 7절 기준으로는 "차이 없음"입니다.**
그러니 이 결과는 "`min_df=5`가 더 좋다"가 아니라 **"피처를 1/5로 줄여도 잃는 것이 없다"** 로 읽어야 합니다.

**그것만으로도 충분히 이득입니다.** ① 학습·예측이 빨라지고(문제 4에서 이 점이 결정적입니다),
② 모델 파일이 작아지고, ③ 드문 조각을 외우는 과적합이 줄어듭니다.
**성능이 같다면 작은 쪽을 고르세요.**

---

## 문제 3. `class_weight="balanced"`

In [ ]:
for cw in [None, "balanced"]:
    m, acc = 학습(X_train, y_train, X_valid, y_valid, class_weight=cw)
    pred = m.predict(X_valid)
    report = classification_report(y_valid, pred, output_dict=True, zero_division=0)
    print(f"class_weight={cw}")
    print(f"  정확도 {acc:.4f} · macro f1 {f1_score(y_valid, pred, average='macro'):.4f}")
    print(f"  사회 재현율 {report['사회']['recall']:.2f} · 사회 정밀도 {report['사회']['precision']:.2f}"
          f" · 세계 재현율 {report['세계']['recall']:.2f}")

**정확도는 내려가고 macro f1은 올라갑니다. 어느 쪽을 볼지 먼저 정해야 하는 상황입니다.**

| | 정확도 | macro f1 | 사회 재현율 | 사회 정밀도 | 세계 재현율 |
|---|---|---|---|---|---|
| 기본 | **0.8455** | 0.8302 | 0.55 | **0.70** | **0.88** |
| `balanced` | 0.8440 | **0.8321** | **0.62** | 0.63 | 0.84 |

`class_weight="balanced"`는 **건수가 적은 주제의 오답에 더 큰 벌점**을 매깁니다.
모델은 `사회`를 더 자주 예측하게 되고, 실제로 놓치는 것이 줄어듭니다(재현율 0.55 → 0.62).
대신 **`사회`가 아닌 것을 `사회`라고 부르는 일이 늘어납니다**(정밀도 0.70 → 0.63).
그 여파로 다른 주제의 재현율이 조금씩 깎입니다(`세계` 0.88 → 0.84).

**놓친 것(재현율)을 줄이는 대신 잘못 넣는 것(정밀도)이 늘어나는 거래**입니다. 공짜가 아닙니다.

**어느 쪽을 골라야 할까요? 용도가 정합니다.**

- **모든 기사를 어딘가에 배정하기만 하면 되고, 전체 오분류 건수가 중요하다** → 기본
- **주제별 담당자가 따로 있어서, 어느 한 주제가 유독 잘 안 잡히면 곤란하다** → `balanced`

**지표를 먼저 정하고 옵션을 고르는 것**이지, 그 반대가 아닙니다.
옵션을 이것저것 켜보고 가장 잘 나온 지표를 골라 보고하는 것은 자기기만입니다.

> **여기서는 어느 쪽도 크게 이기지 못했습니다.** 두 지표의 차이가 0.002 수준이라,
> 본문 7절의 기준으로는 둘 다 "차이 없음"입니다. `사회`가 잘 안 잡히는 진짜 원인은
> 가중치가 아니라 **그 범주의 정의가 모호하다는 것**이라(본문 3절), 손잡이를 돌려서 풀 문제가 아닙니다.

---

## 문제 4. `GridSearchCV`로 벡터화 설정과 `C` 함께 탐색

문자 n-gram은 피처가 13만 개라 그대로 탐색하면 아주 느립니다.
문제 2에서 확인한 `min_df=5`를 고정하면 피처가 2.4만 개로 줄어 **탐색이 5배 빨라집니다.**

In [ ]:
pipe = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", min_df=5),   # ngram_range는 탐색 대상이라 비워둡니다
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
)

param_grid = {
    "tfidfvectorizer__ngram_range": [(2, 3), (2, 4), (1, 3)],
    "logisticregression__C": [1, 5, 20],
}

grid = GridSearchCV(pipe, param_grid, cv=3, scoring="accuracy", n_jobs=-1)
grid.fit(X_train, y_train)   # 검증 데이터는 넣지 않습니다

print("최적 조합:", grid.best_params_)
print("교차 검증 점수(best_score_): %.4f" % grid.best_score_)
print("검증 세트 점수            : %.4f" % accuracy_score(y_valid, grid.predict(X_valid)))
print("본문 8절 기본 설정        : %.4f" % 기준정확도)

**`Pipeline`의 파라미터 이름은 `단계이름__파라미터`입니다.** `make_pipeline`이 붙이는 단계 이름은
클래스명을 소문자로 바꾼 것(`tfidfvectorizer`, `logisticregression`)이고,
`pipe.get_params().keys()`로 확인할 수 있습니다.

**결과를 보면 탐색으로 얻은 것이 없습니다.** 최적 조합이 `ngram_range=(2,3)`, `C=1` —
**우리가 이미 쓰고 있던 기본값**입니다. 검증 점수 0.8468도 기본 설정 0.8455와 사실상 같습니다
(차이 0.0013, 본문 7절 기준으로 "차이 없음").

이 문제의 핵심은 그다음입니다. **`best_score_`(0.8279)가 검증 세트 점수(0.8468)보다 낮습니다.**

`best_score_`를 최종 성능으로 보고하면 안 되는 이유가 여기 있습니다.
**두 숫자는 애초에 다른 조건에서 측정됐습니다.**

| | 무엇을 쟀나 |
|---|---|
| `best_score_` 0.8279 | `cv=3`이므로 **학습 데이터의 2/3(10,667건)로 학습**한 모델 3개의 평균 |
| 검증 세트 0.8468 | 최적 조합으로 **16,000건 전부로 다시 학습**한 모델 |

교차 검증 쪽이 학습 데이터를 적게 쓰니 점수가 낮게 나오는 것이 당연합니다.
반대 방향으로 어긋나는 경우도 흔합니다 — 조합을 많이 탐색할수록 `best_score_`는
**"여러 번 던져서 가장 잘 나온 값"** 이라 낙관적으로 부풀기 때문입니다.

**어느 쪽이든 결론은 같습니다. 탐색에 쓴 점수는 최종 성능이 아닙니다.**
최종 성능은 **탐색에 한 번도 쓰이지 않은 데이터**에서 다시 재야 합니다(본문 12절의 테스트 세트).

> **성능이 거의 오르지 않은 것도 눈여겨보세요.** 본문 11절에서 확인한 대로 남은 오답 대부분이
> **사람도 갈리는 경계**에 있습니다. `C` 값이나 n-gram 범위로는 손댈 수 없는 종류입니다.
> **하이퍼파라미터 탐색은 오답의 성격을 확인한 뒤에** 할 일입니다. 순서를 바꾸면 시간만 씁니다.

---

## 문제 5. 데이터를 늘리면

In [ ]:
for n in [5_000, 20_000, len(전체)]:
    d = 전체.sample(n, random_state=RANDOM_STATE) if n < len(전체) else 전체
    xa, xb, ya, yb = train_test_split(
        d["title"], d["label"], test_size=0.2, stratify=d["label"], random_state=RANDOM_STATE
    )
    _, acc = 학습(xa, ya, xb, yb)
    print(f"학습 {len(xa):>6,}건 → 검증 정확도 {acc:.4f}")

| 학습 데이터 | 검증 정확도 | 직전 대비 |
|---|---|---|
| 4,000건 | 0.7950 | — |
| 16,000건 (본문) | 0.8455 | **+0.051** (4배) |
| 36,542건 (전체) | 0.8577 | **+0.012** (2.3배) |

**데이터를 4배로 늘려 +5.1%p, 거기서 다시 2.3배로 늘려 +1.2%p.**
전형적인 수확 체감입니다. 늘릴수록 이득이지만, **같은 이득을 얻으려면 점점 더 많이 늘려야 합니다.**

본문 8절에서 문자 n-gram으로 얻은 것은 **+8.2%p**였습니다. 데이터를 9배(4,000 → 36,542)로
늘려 얻은 것이 +6.3%p이니, **인자 두 개를 바꾼 쪽이 데이터를 9배 모은 것보다 이득이 컸습니다.**

**이것이 이 노트북 전체의 요지입니다.** 성능을 올릴 방법은 여러 가지이고, 그중에는
**비용이 거의 없는 것과 아주 비싼 것**이 섞여 있습니다.

| 방법 | 얻은 것 | 비용 |
|---|---|---|
| 문자 n-gram으로 바꾸기 | +0.082 | 인자 두 개 |
| 데이터 9배 모으기 | +0.063 | 라벨링 비용, 학습 시간 3배 |
| 전처리 추가 | +0.004 | 함수 하나 (문자 n-gram에서는 오히려 −0.017) |
| 모델 교체 | +0.004 | 없음 |
| 하이퍼파라미터 탐색 | +0.001 | 탐색 시간 |

**싼 것부터 시도하세요.** 표의 위에서 아래로 내려가는 것이 올바른 순서입니다.

> 그리고 이 표의 어떤 숫자도 본문 12절에서 확인한 **분포 이동으로 잃은 8%p**를 상쇄하지 못합니다.
> 배포 환경의 데이터가 학습 데이터와 다르다면, **그 분포의 데이터를 조금 모으는 것**이
> 여기 있는 모든 방법보다 큰 효과를 냅니다.

---

## 문제 6. 정보는 제목의 앞에 있나 뒤에 있나

In [ ]:
def 앞3(s):
    return " ".join(s.split()[:3])


def 뒤3(s):
    return " ".join(s.split()[-3:])


for 이름, 변환 in [("전체", lambda s: s), ("앞 3단어", 앞3), ("뒤 3단어", 뒤3)]:
    _, acc = 학습(X_train.map(변환), y_train, X_valid.map(변환), y_valid)
    print(f"{이름:<8} {acc:.4f}")

print("\n예:", X_train.iloc[0])
print("  앞 3단어:", 앞3(X_train.iloc[0]))
print("  뒤 3단어:", 뒤3(X_train.iloc[0]))

**정보는 압도적으로 앞쪽에 있습니다.**

| | 정확도 |
|---|---|
| 전체 (평균 6.6단어) | 0.8455 |
| **앞 3단어만** | **0.7825** |
| 뒤 3단어만 | 0.6285 |

앞 3단어만 남겨도 0.7825입니다. 절반 이하로 잘라내고도 전체의 93%에 해당하는 성능이 나옵니다.
반면 뒤 3단어만 남기면 0.6285로 무너집니다. **같은 3단어인데 15%p가 넘게 차이 납니다.**

**뉴스 제목의 구조 때문입니다.**

```
국방부 北 풍계리 핵실험장 계속 감시중…준비는 완료돼
└── 주체·소재 ──┘ └────── 부연 설명 ──────┘
```

기사의 **주체와 소재가 앞에** 옵니다. `국방부`, `北`이 나오면 그것만으로 `정치`쪽이라는 신호가
충분합니다. 뒤쪽은 `…` 다음에 붙는 부연이라 주제를 가리키는 힘이 약합니다.

**02번에서 이 결과가 쓰입니다.** `output_sequence_length`(시퀀스 길이)를 정할 때
"뒤를 자르면 얼마나 잃는가"가 곧 이 실험의 답입니다. 여기서 앞쪽이 훨씬 중요하다는 것을
확인했으니, **뒤를 조금 자르는 것은 비교적 안전**합니다.

**다만 이 결론은 이 데이터에만 해당합니다.** 정보가 뒤에 오는 텍스트도 많습니다 —
리뷰의 결론("...하지만 결국 만족합니다"), 부정 표현("좋지 **않았다**")이 그렇습니다.
그런 데이터에서 뒤를 자르면 치명적입니다. **자르기 전에 어디에 정보가 있는지 재보세요.**
그 방법이 방금 한 실험입니다.

---

## 정리

| 문제 | 배운 것 |
|---|---|
| 1 | 정확도는 **어떤 대상을 채점했는지**에 따라 달라진다. 어려운 클래스를 빼면 모델을 안 고쳐도 오른다 |
| 2 | `min_df`로 **성능은 유지하면서 피처를 1/5로 줄일 수 있다.** 같으면 작은 쪽 |
| 3 | `class_weight="balanced"`는 재현율과 정밀도를 맞바꾼다. **지표를 먼저 정한다** |
| 4 | `best_score_`는 **다른 조건에서 잰 숫자**다. 최종 성능은 탐색에 안 쓴 데이터에서 다시 잰다 |
| 5 | 데이터를 늘리면 이득이지만 **수확 체감**한다. 싼 방법부터 시도하는 것이 순서다 |
| 6 | 정보가 어디에 있는지 **재보고** 자른다. 이 데이터는 앞쪽에 몰려 있었다 |